# 00 EDA

Use this notebook after the official Phase 2 data is released. Do not create fake data here; update `configs/baseline.yaml` and place official CSV files under `data/raw/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.utils import load_config
from src.data import load_competition_data, summarize_dataframe
from src.features import infer_column_types

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
config = load_config(CONFIG_PATH)
config

## Load Data

In [ ]:
data = load_competition_data(config)
train = data["train"]
test = data["test"]
sample_submission = data["sample_submission"]

target_col = config["data"]["target_col"]
id_col = config["data"]["id_col"]

train.shape, test.shape, sample_submission.shape

## Shape, Dtypes, Missing Values

In [ ]:
summaries = {
    "train": summarize_dataframe(train, "train"),
    "test": summarize_dataframe(test, "test"),
    "sample_submission": summarize_dataframe(sample_submission, "sample_submission"),
}
summaries

In [ ]:
missing = pd.concat(
    [train.isna().mean().rename("train_missing_rate"), test.isna().mean().rename("test_missing_rate")],
    axis=1,
).sort_values("train_missing_rate", ascending=False)
missing.head(30)

## Target Check

In [ ]:
train[target_col].describe(include="all")

In [ ]:
if config["task"]["problem_type"] == "classification":
    display(train[target_col].value_counts(dropna=False).head(30))
else:
    train[target_col].hist(bins=50)
    plt.title("Target distribution")
    plt.show()

## Duplicate Rows

In [ ]:
train.duplicated().sum(), test.duplicated().sum(), train[id_col].duplicated().sum(), test[id_col].duplicated().sum()

## Column Type Inference

In [ ]:
column_types = infer_column_types(
    train,
    target_col=target_col,
    id_col=id_col,
    parse_datetime=config.get("features", {}).get("parse_datetime", True),
)
column_types

## Train/Test Distribution Shift

In [ ]:
numeric_cols = column_types["numeric"]
shift_rows = []
for col in numeric_cols:
    shift_rows.append({
        "column": col,
        "train_mean": train[col].mean(),
        "test_mean": test[col].mean(),
        "mean_diff": abs(train[col].mean() - test[col].mean()),
        "train_missing": train[col].isna().mean(),
        "test_missing": test[col].isna().mean(),
    })
pd.DataFrame(shift_rows).sort_values("mean_diff", ascending=False).head(30)

## Adversarial Validation Placeholder

After basic checks, train a simple classifier to distinguish train vs test. High AUC means distribution shift may affect CV reliability. Keep this as a diagnostic, not a leaderboard optimization trick.

In [ ]:
# TODO: Add adversarial validation after confirming columns and leakage rules.
# Recommended: reuse FeatureBuilder, label train rows as 0 and test rows as 1,
# then run cross_val_score with roc_auc.

## EDA Hypotheses

Write five concrete hypotheses in `reports/eda_report.md` before feature engineering.